# 03 — CNN face-verification experiments & threshold lock

So sánh hai checkpoint của **FaceNet Inception-ResNet-v1 CNN** (`VGGFace2`, `CASIA-WebFace`) trên official folds 0–7. Chọn checkpoint bằng cross-validation, rồi khóa cosine threshold trên fold 8 tại target FMR 1%. Fold 9 không được parse hoặc score trong notebook này.


In [1]:
import os
import sys
import json
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn>=1.5", "pandas>=2.2", "seaborn>=0.13",
    "opencv-python-headless>=4.10", "tqdm>=4.66",
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter

SEED = 464
random.seed(SEED)
np.random.seed(SEED)

# A blank Colab runtime is the reference environment.  Nothing from src/ or
# scripts/ is needed.  All intermediate DS assets live under this runtime root.
RUNTIME_ROOT = Path("/content/facekyc_ds") if Path("/content").exists() else Path.cwd() / ".facekyc_ds"
DATA_ROOT = RUNTIME_ROOT / "data"
REPORT_ROOT = RUNTIME_ROOT / "reports"
ARTIFACT_ROOT = RUNTIME_ROOT / "artifacts"
for directory in (DATA_ROOT, REPORT_ROOT, ARTIFACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("Runtime root:", RUNTIME_ROOT)
print("Python:", sys.version.split()[0])


Runtime root: /content/facekyc_ds
Python: 3.13.15


In [2]:
from sklearn.datasets import fetch_lfw_people

# This call downloads the official funneled LFW archive plus pair protocols.
_ = fetch_lfw_people(
    data_home=str(DATA_ROOT), color=True, resize=0.5,
    min_faces_per_person=0, download_if_missing=True,
)
LFW_HOME = DATA_ROOT / "lfw_home"
LFW_IMAGE_ROOT = LFW_HOME / "lfw_funneled"
assert LFW_IMAGE_ROOT.exists(), f"LFW extraction failed: {LFW_IMAGE_ROOT}"

def subject_bucket(subject):
    # Stable subject-level split: 80% train, 10% validation, 10% locked holdout.
    return int(hashlib.sha256(subject.encode("utf-8")).hexdigest()[:8], 16) % 10

def build_lfw_manifest():
    rows = []
    for path in sorted(LFW_IMAGE_ROOT.glob("*/*.jpg")):
        subject = path.parent.name
        bucket = subject_bucket(subject)
        split = "train" if bucket < 8 else ("validation" if bucket == 8 else "holdout")
        rows.append({"path": str(path), "subject": subject, "split": split})
    frame = pd.DataFrame(rows)
    assert len(frame) == 13233, f"Expected 13,233 LFW images, found {len(frame)}"
    assert frame.groupby("subject")["split"].nunique().max() == 1
    return frame

MANIFEST_PATH = DATA_ROOT / "lfw_subject_manifest.csv"
manifest = build_lfw_manifest()
manifest.to_csv(MANIFEST_PATH, index=False)
print("LFW images:", len(manifest), "subjects:", manifest.subject.nunique())


LFW images: 13233 subjects: 5749


In [3]:
from sklearn.metrics import roc_auc_score

def verification_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    accepted = scores >= threshold
    impostor = labels == 0
    genuine = labels == 1
    return {
        "threshold": float(threshold),
        "fmr": float(accepted[impostor].mean()),
        "fnmr": float((~accepted[genuine]).mean()),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, scores)),
        "pairs": int(len(labels)),
    }

def threshold_for_fmr(labels, scores, target_fmr=0.01):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    impostor_scores = np.sort(scores[labels == 0])
    rank = int(np.ceil((1.0 - target_fmr) * len(impostor_scores))) - 1
    rank = int(np.clip(rank, 0, len(impostor_scores) - 1))
    return float(np.nextafter(impostor_scores[rank], np.inf))

def pad_metrics(labels, live_scores, threshold):
    labels = np.asarray(labels).astype(int)  # 0=attack, 1=bona fide
    live_scores = np.asarray(live_scores, dtype=float)
    accepted = live_scores >= threshold
    attack = labels == 0
    bona_fide = labels == 1
    apcer = float(accepted[attack].mean())
    bpcer = float((~accepted[bona_fide]).mean())
    return {
        "threshold": float(threshold), "apcer": apcer, "bpcer": bpcer,
        "acer": float((apcer + bpcer) / 2),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, live_scores)),
        "samples": int(len(labels)),
    }

def threshold_for_apcer(labels, live_scores, target_apcer=0.10):
    labels = np.asarray(labels).astype(int)
    attack_scores = np.sort(np.asarray(live_scores, dtype=float)[labels == 0])
    rank = int(np.ceil((1.0 - target_apcer) * len(attack_scores))) - 1
    rank = int(np.clip(rank, 0, len(attack_scores) - 1))
    return float(np.nextafter(attack_scores[rank], np.inf))


In [4]:
def pair_image_path(subject, one_based_index):
    return LFW_IMAGE_ROOT / subject / f"{subject}_{int(one_based_index):04d}.jpg"

def parse_pair_line(line):
    parts = line.strip().split("\t")
    if len(parts) == 3:
        return pair_image_path(parts[0], parts[1]), pair_image_path(parts[0], parts[2]), 1
    if len(parts) == 4:
        return pair_image_path(parts[0], parts[1]), pair_image_path(parts[2], parts[3]), 0
    raise ValueError(f"Malformed LFW pair line: {line!r}")

def load_official_fold_records(folds):
    # Parse only the requested folds. Fold 9 remains untouched until notebook 05.
    protocol_path = LFW_HOME / "pairs.txt"
    with protocol_path.open(encoding="utf-8") as handle:
        header = next(handle).strip()
        assert header == "10\t300", header
        requested = set(folds)
        records = []
        for row_index, line in enumerate(handle):
            fold = row_index // 600
            if fold not in requested:
                continue
            left, right, label = parse_pair_line(line)
            assert left.exists() and right.exists()
            records.append({"left": str(left), "right": str(right), "label": label, "fold": fold})
    return pd.DataFrame(records)


In [5]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch==2.6.0"
])
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

class PairDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        with Image.open(row.left) as image:
            left = self.transform(image.convert("RGB"))
        with Image.open(row.right) as image:
            right = self.transform(image.convert("RGB"))
        return left, right, int(row.label), int(row.fold)

def build_backbone(name):
    if name not in {"vggface2", "casia-webface"}:
        raise ValueError(name)
    model = InceptionResnetV1(pretrained=name, classify=False).eval().to(DEVICE)

    def transform(image):
        # fixed_image_standardization expects float pixels in [0, 255].
        # torchvision.transforms.ToTensor() would first scale them to [0, 1]
        # and collapse FaceNet embeddings to an almost constant vector.
        resized = image.resize((160, 160), Image.Resampling.BILINEAR)
        pixels = np.asarray(resized, dtype=np.float32)
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        return fixed_image_standardization(tensor)

    return model, transform

@torch.inference_mode()
def score_pairs(model_name, records, batch_size=64):
    model, transform = build_backbone(model_name)
    loader = DataLoader(PairDataset(records, transform), batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=DEVICE.type == "cuda")
    scores, labels, folds = [], [], []
    for left, right, label, fold in loader:
        left = F.normalize(model(left.to(DEVICE, non_blocking=True)), dim=1)
        right = F.normalize(model(right.to(DEVICE, non_blocking=True)), dim=1)
        scores.extend((left * right).sum(dim=1).cpu().numpy().tolist())
        labels.extend(label.numpy().tolist())
        folds.extend(fold.numpy().tolist())
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return np.asarray(scores), np.asarray(labels), np.asarray(folds)


Device: cuda Tesla T4


In [6]:
assert DEVICE.type == "cuda", "Notebook 03 nên chạy bằng Colab GPU."
records = load_official_fold_records(range(9))
assert set(records.fold.unique()) == set(range(9))
print(records.groupby(["fold", "label"]).size().unstack())

candidate_scores = {}
for name in ["vggface2", "casia-webface"]:
    scores, labels, folds = score_pairs(name, records)
    candidate_scores[name] = {"scores": scores, "labels": labels, "folds": folds}


label    0    1
fold           
0      300  300
1      300  300
2      300  300
3      300  300
4      300  300
5      300  300
6      300  300
7      300  300
8      300  300


  0%|          | 0.00/107M [00:00<?, ?B/s]

  0%|          | 0.00/111M [00:00<?, ?B/s]

In [7]:
scorecard = []
for name, values in candidate_scores.items():
    fold_metrics = []
    for held_out_fold in range(8):
        fit = (values["folds"] < 8) & (values["folds"] != held_out_fold)
        assess = values["folds"] == held_out_fold
        threshold = threshold_for_fmr(values["labels"][fit], values["scores"][fit], 0.01)
        fold_metrics.append(verification_metrics(values["labels"][assess], values["scores"][assess], threshold))
    summary = {
        "backbone": name,
        "mean_fmr": float(np.mean([row["fmr"] for row in fold_metrics])),
        "mean_fnmr": float(np.mean([row["fnmr"] for row in fold_metrics])),
        "mean_accuracy": float(np.mean([row["accuracy"] for row in fold_metrics])),
        "mean_auc": float(np.mean([row["auc"] for row in fold_metrics])),
        "std_accuracy": float(np.std([row["accuracy"] for row in fold_metrics])),
        "fold_metrics": fold_metrics,
    }
    scorecard.append(summary)

comparison = pd.DataFrame([{k: v for k, v in row.items() if k != "fold_metrics"} for row in scorecard])
comparison = comparison.sort_values(["mean_accuracy", "mean_auc"], ascending=False)
display(comparison)
selected = comparison.iloc[0].backbone
print("Selected CNN backbone:", selected)


,backbone,mean_fmr,mean_fnmr,mean_accuracy,mean_auc,std_accuracy
0,vggface2,0.01,0.067083,0.961458,0.992685,0.004672
1,casia-webface,0.01,0.687500,0.651250,0.885868,0.015784


Selected CNN backbone: vggface2


In [8]:
chosen = candidate_scores[selected]
validation_mask = chosen["folds"] == 8
locked_threshold = threshold_for_fmr(
    chosen["labels"][validation_mask], chosen["scores"][validation_mask], 0.01
)
validation_metrics = verification_metrics(
    chosen["labels"][validation_mask], chosen["scores"][validation_mask], locked_threshold
)
report = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "LFW official 10-fold protocol",
    "candidate_backbones": ["vggface2", "casia-webface"],
    "selection_folds": list(range(8)),
    "validation_fold": 8,
    "holdout_fold": 9,
    "target_fmr": 0.01,
    "selected_backbone": selected,
    "locked_threshold": locked_threshold,
    "scorecard": scorecard,
    "validation_metrics": validation_metrics,
    "holdout_accessed": False,
    "limitation": "FaceNet checkpoints are research baselines; target-domain ID/selfie validation is still required.",
}
output = REPORT_ROOT / "03_verification_selection.json"
output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
report


{'created_at': '2026-08-31T04:19:08.372791+00:00',
 'dataset': 'LFW official 10-fold protocol',
 'candidate_backbones': ['vggface2', 'casia-webface'],
 'selection_folds': [0, 1, 2, 3, 4, 5, 6, 7],
 'validation_fold': 8,
 'holdout_fold': 9,
 'target_fmr': 0.01,
 'selected_backbone': 'vggface2',
 'locked_threshold': 0.4225261211395264,
 'scorecard': [{'backbone': 'vggface2',
   'mean_fmr': 0.01,
   'mean_fnmr': 0.06708333333333333,
   'mean_accuracy': 0.9614583333333333,
   'mean_auc': 0.9926847222222221,
   'std_accuracy': 0.004672429477501219,
   'fold_metrics': [{'threshold': 0.42175486683845526,
     'fmr': 0.01,
     'fnmr': 0.07333333333333333,
     'accuracy': 0.9583333333333334,
     'auc': 0.9874333333333333,
     'pairs': 600},
    {'threshold': 0.42261135578155523,
     'fmr': 0.006666666666666667,
     'fnmr': 0.07666666666666666,
     'accuracy': 0.9583333333333334,
     'auc': 0.9948999999999999,
     'pairs': 600},
    {'threshold': 0.4202274680137635,
     'fmr': 0.013333

## Kết luận sau khi chạy

Candidate và threshold đã khóa trước holdout. Không được đổi checkpoint/threshold sau khi xem notebook 05. FaceNet là CNN face-specific nhưng LFW vẫn không đại diện ảnh giấy tờ/selfie Việt Nam; vòng dữ liệu sau cần benchmark đúng miền dữ liệu có consent.
